In [32]:
#the import packages
import requests
import pandas as pd
from pandas import json_normalize
import requests
import os
from pathlib import Path
from datetime import datetime, timezone,timedelta

import json
import numpy as np

In [40]:
base_url = "http://personal-laptop:8080/"


In [34]:
def create_url(base_url,first_part_endpoint,second_part_endpoint):
    return base_url + first_part_endpoint + second_part_endpoint
def getData(url):
    response = requests.get(url)

    # Check response status
    if response.status_code == 200:
        
        data = response.json()
        print(f"Received {len(data)} records.")
        return data
    else:
        print(f"Error: {response.status_code}")


def saveDataIntoDataFolder(data,data_file_name):
    script_dir = Path().resolve().parent
    data_folder = script_dir / 'dataAnalysis and machine learning'/'data'
    print(data_folder)
    data_folder.mkdir(exist_ok=True)
    
    file_path = data_folder / (data_file_name + ".json")
    with open(file_path, 'w') as file:
        if isinstance(data, pd.DataFrame):
            print("It's a DataFrame")
            if  data.empty:
                print("No data to save.")
            else:
                data.to_json(file_path, orient='records', lines=False)             
                print(f"Data saved to {file_path}")

        else:  
            print("It's NOT a DataFrame.")    
            if not data:
                print("No data to save.")
            else:    
                json.dump(data, file)
                print(f"Data saved to {file_path}")

        
def getDataFromServerThenSaveThemIntoDataFolder(url,data_file_name):
    data = getData(url)
    saveDataIntoDataFolder(data,data_file_name)
    
def loadDataFromFile(file_name):
    script_dir = Path().resolve().parent

    data_folder = script_dir / 'dataAnalysis and machine learning'/'data'
    print(data_folder)
    data_folder.mkdir(exist_ok=True)
    
    file_path = data_folder / (file_name + ".json")
    
    if file_path.exists():
        df = pd.read_json(file_path)
        print(f"Loaded {len(df)} records from {file_path}")
        return df
    else:
        print(f"File {file_path} does not exist.")
        return None    

In [35]:
def getTimeSeries(df_userInputData_smallScale):
    first_row= df_userInputData_smallScale.iloc[0]
    last_row = df_userInputData_smallScale.iloc[-1]
    start_time = first_row["epoch_ms"]
    end_time   = last_row["epoch_ms"] 
    first_part_endpoint = "timeSeriesEndpoints/"
    second_part_endpoint = "getTimeSeriesData?start=" + str(start_time) + '&end=' +str(end_time)
    url = create_url(base_url,first_part_endpoint,second_part_endpoint)
    data = getData(url)
    df_timeSeries = pd.DataFrame(data)

    return df_timeSeries
def tranformTimeSeries(df_timeSeries):
    # 1. Select only the needed column
    df_timeSeriesExperiments = df_timeSeries[['timestamp', 'Id', 'BME680:breathVocEquivalent']].copy()
    # 2. Convert `timestamp` (UTC) to Europe/Athens tz-aware datetime
    df_timeSeriesExperiments['timestamp'] = (
        pd.to_datetime(df_timeSeriesExperiments['timestamp'], utc=True)
          .dt.tz_convert('Europe/Athens')
    )
    # 3. Create the two “Id=…:” columns
    df_timeSeriesExperiments['Id=0:BME680:breathVocEquivalent'] = np.where(
        df_timeSeriesExperiments['Id'] == 0, df_timeSeriesExperiments['BME680:breathVocEquivalent'], np.nan
    )
    df_timeSeriesExperiments['Id=1:BME680:breathVocEquivalent'] = np.where(
        df_timeSeriesExperiments['Id'] == 1, df_timeSeriesExperiments['BME680:breathVocEquivalent'], np.nan
    )
    df_timeSeriesExperiments['Id=2:BME680:breathVocEquivalent'] = np.where(
        df_timeSeriesExperiments['Id'] == 2, df_timeSeriesExperiments['BME680:breathVocEquivalent'], np.nan
    )
    
    # 4. Set timestamp as the row index
    #df_timeSeriesExperiments.set_index('timestamp', inplace=True)
    
    # 5. Drop the old Id+raw measurement columns
    df_timeSeriesExperiments = df_timeSeriesExperiments.drop(
        columns=['Id', 'BME680:breathVocEquivalent']
    )
    return df_timeSeriesExperiments

def getAndTransformTimeSeries(df_userInputData_smallScale):
    df_timeSeries = getTimeSeries(df_userInputData_smallScale)  
    df_timeSeriesExperiments = tranformTimeSeries(df_timeSeries)
    return df_timeSeriesExperiments

In [36]:
def filterUserInputData(dataframe,fields,comparisons,conditions="NaN",oldData = False,experimentStateSpecified =["InsertingSourcePollutant"]):
# Define the condition
    field = fields[0]
    comparison = comparisons[0]
    if comparison == "equals":
        
        final_condition = (dataframe[field].isin(conditions)) & (dataframe["experimentState"].isin(experimentStateSpecified))
    if comparison == "contains" and conditions != "NaN":
        final_condition = (
            dataframe[field].str.contains("|".join(conditions), na=False)
            & dataframe["experimentState"].isin(experimentStateSpecified)
        )
        final_condition = (final_condition) & (dataframe["experimentState"].isin(experimentStateSpecified))    
    if comparison == "not-equals":
        final_condition = (~(dataframe[field].isin(conditions))& (dataframe["experimentState"].isin(experimentStateSpecified)))     
    if comparison =="NaN":    
        final_condition = ((dataframe[field].isna()) & (dataframe["experimentState"].isin(experimentStateSpecified)))
    # Get the indices of the rows that match the condition
  #  print(final_condition)
    matching_indices = dataframe[final_condition].index
    # Initialize a list to collect the desired indices
    all_indices = []
    # Add previous, current, and next indices (if within bounds)
    for idx in matching_indices:
        if oldData == False:
            if idx - 1 >= 0 and idx + 1 < len(dataframe) :
            
                if (dataframe.iloc[idx-1]["experimentState"]=="StartingExperiment") and  (dataframe.iloc[idx+1]["experimentState"]=="RemovingSourcePollutant"):

                    all_indices.append(idx - 1)
                    all_indices.append(idx)
                    all_indices.append(idx + 1)
                #when we didn't use the start experiment state for all the experiments,only the inserting source and remove source
        elif oldData == True: 
            if idx + 1 < len(dataframe) :  
           
                if dataframe.iloc[idx+1]["experimentState"]=="RemovingSourcePollutant":
                    all_indices.append(idx)
                    all_indices.append(idx + 1) 
            
    # Remove duplicates and preserve order
    # Use dict.fromkeys to maintain order while removing duplicates
    all_indices = list(dict.fromkeys(all_indices))
    
    # Filter the DataFrame
    df_userInputData_filtered= dataframe.loc[all_indices]
    df_userInputData_filtered=df_userInputData_filtered.reset_index(drop = True)
    print(df_userInputData_filtered.shape)
    return df_userInputData_filtered

In [37]:
def specifyUserInputDataWanted(df_userInputData_smallScale,field,comparison,conditions,oldData = False,experimentStateSpecified = ["InsertingSourcePollutant"]):
 

  
    df_userInputData_smallScale = filterUserInputData(df_userInputData_smallScale,field,comparison,conditions,oldData,experimentStateSpecified)

    
    df_userInputData_smallScale = df_userInputData_smallScale.reset_index(drop=True) 

    return df_userInputData_smallScale

In [38]:


def takeTheUserInputDataFromParticularRoom(df_userInputData,particularRoom,oldData = False,experimentStateSpecified =  ["InsertingSourcePollutant"]):
    df_userInputData_smallScale = filterUserInputData(df_userInputData,"room","equals",particularRoom,oldData,experimentStateSpecified)

    df_userInputData_smallScale = filterUserInputData(df_userInputData_smallScale,"item-used","equals","Φαρμακευτικό αλκοόλ 95%",oldDataa,experimentStateSpecified)

    df_userInputData_smallScale = filterUserInputData(df_userInputData_smallScale,"are-fans-on","NaN",oldData,experimentStateSpecified)

    df_userInputData_smallScale = filterUserInputData(df_userInputData_smallScale,"are-windows-opened","NaN",oldData,experimentStateSpecified)

    df_userInputData_smallScale = filterUserInputData(df_userInputData_smallScale,"notes","not-equals","ανεμιστήρας",oldData,experimentStateSpecified)

    df_userInputData_smallScale = df_userInputData_smallScale.reset_index(drop=True) 

    return df_userInputData_smallScale
    
def countPositionOfPollutantSource(df_userInputData_smallScale):
    df_unique_position_count = df_userInputData_smallScale.groupby(['front-wall', 'side-right-wall','back-wall','side-left-wall'], dropna=False).size().reset_index(name='count')
    return df_unique_position_count    
    

In [41]:
first_part_endpoint = "dataAnalysisEndpoints/"
second_part_endpoint = "getAllUserInputsExperimentState"        

url = create_url(base_url,first_part_endpoint,second_part_endpoint)


df_userInputData = getData(url)
df_userInputData =pd.DataFrame(df_userInputData)
if 'details' in df_userInputData.columns:
    details_df = df_userInputData["details"].apply(pd.Series)
    
    # Then join with the original DataFrame (drop the nested 'details' if desired)
    df_userInputData = pd.concat([df_userInputData.drop(columns=["details"]), details_df], axis=1)

if 0 in df_userInputData.columns:
    df_userInputData =df_userInputData.drop(columns=[0])    

# Convert epoch (assumes seconds) to datetime in local time
df_userInputData["epoch_ms"] = df_userInputData["timestamp"].apply(lambda x: x["$date"])
df_userInputData["timestamp_local"] = pd.to_datetime(df_userInputData["epoch_ms"], unit="ms").dt.tz_localize("UTC").dt.tz_convert("Europe/Athens")
df_userInputData.drop(columns = ["timestamp"])
dimensions = ['front-wall','side-right-wall','back-wall','side-left-wall']
for dimension in dimensions:
    df_userInputData[dimension] = pd.to_numeric(df_userInputData[dimension], errors='coerce')


ConnectionError: HTTPConnectionPool(host='personal-laptop', port=8080): Max retries exceeded with url: /dataAnalysisEndpoints/getAllUserInputsExperimentState (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x0000028EF5E30D10>: Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it'))

In [ ]:
df_userInputData

In [ ]:
print(df_userInputData["room"].unique())
saveDataIntoDataFolder(df_userInputData,"All User Input Data")

In [ ]:
df_userInputData = loadDataFromFile("All User Input Data")
df_userInputData

In [ ]:
df_userInputData = loadDataFromFile("All User Input Data")
oldData = False
experimentStateSpecified =["NoSourcePollutantInserted","InsertingSourcePollutant"]
fields = ["room"]
comparisons = ["contains"]
conditions = ['Όλοι οι αισθητήρες μαζί Μ:0.80 , Α:0.90','Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55','Κρεβατοκάμαρα id:1 Π0.6 Α1.2 id:2 Μ0.8 Α1.1']
df_userInputData_smallScale = specifyUserInputDataWanted(df_userInputData,fields,comparisons,conditions,oldData,experimentStateSpecified)


In [ ]:
df_userInputData_smallScale

In [ ]:
transformSeries = getAndTransformTimeSeries(df_userInputData_smallScale)
transformSeries

In [ ]:
transformSeries.head(20)

In [ ]:


saveDataIntoDataFolder(df_userInputData_smallScale,"User:Previous experiments")
saveDataIntoDataFolder(transformSeries,"Data:Previous experiments")